# Fruit Detection + Defect Measurement + MQTT

Notebook này gồm:
- Nhận diện trái cây (Edge Impulse)
- Đo tỷ lệ vết thâm
- Hiển thị FPS
- Gửi dữ liệu qua MQTT

In [ ]:
# =========================
# IMPORT
# =========================
from picamera2 import Picamera2
import cv2
import numpy as np
from edge_impulse_linux.image import ImageImpulseRunner
import time
import paho.mqtt.client as mqtt

In [ ]:
# =========================
# MQTT CONFIG
# =========================
MQTT_BROKER = "192.168.1.194"
MQTT_TOPIC = "esp32/control"

last_sent_time = 0
interval = 0.3

client = mqtt.Client()
client.connect(MQTT_BROKER, 1883, 60)

print("MQTT connected")

In [ ]:
# =========================
# ROI + DEFECT FUNCTION
# =========================
ROI_X, ROI_Y, ROI_W, ROI_H = 80, 120, 480, 350

def measure_defect(frame_bgr):
    roi = frame_bgr[ROI_Y:ROI_Y+ROI_H, ROI_X:ROI_X+ROI_W]
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

    bg_mask = cv2.inRange(hsv, (0, 0, 0), (180, 35, 255))
    belt_mask = cv2.inRange(hsv, (35, 30, 0), (95, 255, 130))

    exclude = cv2.bitwise_or(belt_mask, bg_mask)
    fruit_mask = cv2.bitwise_not(exclude)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    fruit_mask = cv2.morphologyEx(fruit_mask, cv2.MORPH_CLOSE, kernel)
    fruit_mask = cv2.morphologyEx(fruit_mask, cv2.MORPH_OPEN, kernel)

    fruit_area = cv2.countNonZero(fruit_mask)
    if fruit_area < 1000:
        return 0.0, None

    brown_mask = cv2.inRange(hsv, (5, 30, 30), (25, 180, 140))
    dark_mask = cv2.inRange(hsv, (0, 0, 0), (180, 255, 60))

    defect_raw = cv2.bitwise_or(brown_mask, dark_mask)
    defect_mask = cv2.bitwise_and(defect_raw, fruit_mask)

    k_small = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    defect_mask = cv2.morphologyEx(defect_mask, cv2.MORPH_OPEN, k_small)

    defect_area = cv2.countNonZero(defect_mask)
    defect_percent = (defect_area / fruit_area) * 100

    return defect_percent, defect_mask

In [ ]:
# =========================
# LOAD MODEL
# =========================
print("Loading model...")

runner = ImageImpulseRunner("./fruit_detech-linux-aarch64-v2.eim")
model_info = runner.init()

input_width = model_info['model_parameters']['image_input_width']
input_height = model_info['model_parameters']['image_input_height']

print(f"Model ready: {input_width}x{input_height}")

In [ ]:
# =========================
# CAMERA
# =========================
picam2 = Picamera2()
picam2.configure(picam2.create_video_configuration(
    main={"size": (640, 480), "format": "XRGB8888"}
))

picam2.set_controls({
    "AwbEnable": True,
    "AeEnable": True,
    "Brightness": 0.0,
    "Contrast": 1.1,
    "Saturation": 1.3,
    "Sharpness": 1.0
})

picam2.start()
time.sleep(1.0)

print("Camera started")

In [ ]:
# =========================
# MAIN LOOP
# =========================
fps_time = time.time()

try:
    while True:
        frame = picam2.capture_array("main")
        if frame is None:
            continue

        if len(frame.shape) == 3 and frame.shape[2] == 4:
            frame_bgr = frame[:, :, :3].copy()
        else:
            frame_bgr = frame.copy()

        resized = cv2.resize(frame_bgr, (input_width, input_height))
        img_for_model = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)

        features, _ = runner.get_features_from_image(img_for_model)
        result = runner.classify(features)

        now_fps = time.time()
        fps = 1.0 / max(now_fps - fps_time, 0.001)
        fps_time = now_fps

        probs = result['result']['classification']
        label = max(probs, key=probs.get)
        confidence = probs[label]

        if confidence > 0.7:
            now = time.time()
            if now - last_sent_time > interval:
                client.publish(MQTT_TOPIC, label)

        display_frame = frame_bgr.copy()

        cv2.putText(display_frame, f"{label} ({confidence:.2f})", (15, 45),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,255,0), 2)

        cv2.putText(display_frame, f"FPS: {fps:.1f}", (15, 90),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,255,255), 2)

        if label != "no_fruit" and confidence > 0.7:
            defect_pct, defect_mask = measure_defect(frame_bgr)

            color = (0,0,255) if defect_pct > 5 else (0,255,0)
            cv2.putText(display_frame, f"Defect: {defect_pct:.1f}%", (15,135),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 2)

        cv2.rectangle(display_frame, (ROI_X, ROI_Y),
                      (ROI_X + ROI_W, ROI_Y + ROI_H), (255,255,0), 1)

        cv2.imshow("Fruit Detect", display_frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

except KeyboardInterrupt:
    pass

finally:
    runner.stop()
    picam2.stop()
    cv2.destroyAllWindows()
    print("Done")